In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import shutil
import random

# Define paths
base_dir = "C:/tuberculosis/TB_Chest_Radiography_Database"
output_dir = "C:/tuberculosis/dataset"

# Class folders
classes = ["Normal", "Tuberculosis"]

# Train, test, validation split percentages
split_ratios = {"train": 0.7, "test": 0.15, "val": 0.15}

# Create output directories
for split in split_ratios.keys():
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

# Function to split and move files
def split_data(cls):
    cls_path = os.path.join(base_dir, cls)
    images = os.listdir(cls_path)
    random.shuffle(images)
    
    total_images = len(images)
    train_size = int(split_ratios["train"] * total_images)
    test_size = int(split_ratios["test"] * total_images)

    for i, img in enumerate(images):
        if i < train_size:
            split_folder = "train"
        elif i < train_size + test_size:
            split_folder = "test"
        else:
            split_folder = "val"

        src = os.path.join(cls_path, img)
        dest = os.path.join(output_dir, split_folder, cls, img)
        shutil.move(src, dest)

# Apply splitting for both classes
for cls in classes:
    split_data(cls)

print("Dataset organized into train, test, and validation sets.")

Dataset organized into train, test, and validation sets.


In [3]:
import random
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import shutil

train_normal_dir = "C:/tuberculosis/dataset/train/Normal"
backup_dir = "C:/tuberculosis/dataset/extra_normal"

# Create backup folder
os.makedirs(backup_dir, exist_ok=True)

# List all normal images
all_normal_images = os.listdir(train_normal_dir)

# Select 700 random normal images
selected_images = random.sample(all_normal_images, 700)

# Move extra images to backup folder
for img in all_normal_images:
    if img not in selected_images:
        shutil.move(os.path.join(train_normal_dir, img), os.path.join(backup_dir, img))

print("Undersampling completed: Normal images reduced to 700 in train set.")

Undersampling completed: Normal images reduced to 700 in train set.


In [4]:
import os
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img

# Paths
train_tb_dir = "C:/tuberculosis/dataset/train/Tuberculosis"
augmented_tb_dir = "C:/tuberculosis/dataset/augmented_tb"

# Create folder for augmented images
os.makedirs(augmented_tb_dir, exist_ok=True)

# Data augmentation setup
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# List TB images
tb_images = os.listdir(train_tb_dir)

# Generate augmented images
for img_name in tb_images:
    img_path = os.path.join(train_tb_dir, img_name)
    img = load_img(img_path)
    img_array = img_to_array(img)
    img_array = img_array.reshape((1,) + img_array.shape)

    # Create 4 new images per original image
    i = 0
    for batch in datagen.flow(img_array, batch_size=1, save_to_dir=augmented_tb_dir, save_prefix="aug", save_format="jpeg"):
        i += 1
        if i >= 4:  # Stop after generating 4 images per original
            break

print("Augmentation completed: New TB images generated.")

Augmentation completed: New TB images generated.


In [5]:
import shutil
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Paths
augmented_tb_dir = "C:/tuberculosis/dataset/augmented_tb"
train_tb_dir = "C:/tuberculosis/dataset/train/Tuberculosis"

# Move all augmented images to the training folder
for img_name in os.listdir(augmented_tb_dir):
    src = os.path.join(augmented_tb_dir, img_name)
    dest = os.path.join(train_tb_dir, img_name)
    shutil.move(src, dest)

# Remove the now-empty augmented folder
os.rmdir(augmented_tb_dir)

print("Augmented TB images merged into the training dataset.")

Augmented TB images merged into the training dataset.


In [6]:
import tensorflow as tf
print(tf.__version__)

2.18.0


In [7]:
from tensorflow.keras.models import load_model

model = load_model("tb_detection_model.h5")  # Load original model
model.save("tb_model_revised.keras")  # Save in a more universal format